# ¿Se ha parado España? La demanda eléctrica durante el 11N

En este EDA comparo la demanda eléctrica real con la prevista por Red Eléctrica durante la huelga general del 11 de noviembre de 2026. No utilizo la diferencia como porcentaje de seguimiento: la electricidad mide actividad, no personas.

## 1. Preparación

Conservo la respuesta JSON original de REData. Cada serie contiene puntos de cinco minutos aunque la consulta utilice `time_trunc=hour`. Emparejo real y prevista por su fecha exacta.

In [1]:
from analisis_huelga_11n import cargar_series, resumir, guardar

bruto, filas = cargar_series()
print(f'Series comparables: {len(filas)}')
print(f'Actualización de REData: {bruto["data"]["attributes"]["last-update"]}')

Series comparables: 288
Actualización de REData: 2026-11-12T10:07:05.000+01:00


## 2. Diferencia frente a la previsión

Para cada instante calculo `(real - prevista) / prevista`. Una cifra negativa significa que la red registra menos demanda de la esperada. Para aproximar la energía acumulada integro cada diferencia durante cinco minutos.

In [2]:
resumen = resumir(filas)
guardar(filas, resumen)
for clave, valor in resumen.items():
    print(f'{clave}: {valor}')

ultima_actualizacion: 2026-11-11T23:55:00.000+01:00
observaciones: 288
real_mw: 26349.0
prevista_mw: 26340.0
diferencia_mw: 9.0
diferencia_pct: 0.034
mayor_caida_pct: -9.02
momento_mayor_caida: 2026-11-11T08:00:00.000+01:00
deficit_acumulado_mwh: 27591.17
exceso_acumulado_mwh: 153.67
energia_real_mwh: 703416.5
energia_prevista_mwh: 730854.0
diferencia_diaria_mwh: -27437.5
diferencia_diaria_pct: -3.754
provisional: False


## 3. Lectura y límites

Analizo la jornada completa y comparo la demanda real con la previsión de REE, no con el día anterior. Aun así, el desvío puede incluir error de previsión, meteorología, autoconsumo y traslados de consumo por teletrabajo. No lo convierto en una tasa de seguimiento.

In [3]:
peor = min(filas, key=lambda fila: fila['diferencia_pct'])
print('Mayor caída observada durante la jornada:')
print(f"{peor['datetime']}: {peor['diferencia_pct']:.2f}% ({peor['diferencia_mw']:.0f} MW)")
print(f"Balance del día: {resumen['diferencia_diaria_pct']:.2f}% ({resumen['diferencia_diaria_mwh'] / 1000:.2f} GWh) frente a la previsión")
print('Conclusión: describo el desvío eléctrico; no estimo cuántas personas hicieron huelga.')

Mayor caída observada durante la jornada:
2026-11-11T08:00:00.000+01:00: -9.02% (-2850 MW)
Balance del día: -3.75% (-27.44 GWh) frente a la previsión
Conclusión: describo el desvío eléctrico; no estimo cuántas personas hicieron huelga.
